# Granite Guardian 3.2 5B Factuality Correction LoRA: Detailed Guide 

Link to 🤗 model: [Granite-Guardian-3.2-5B](https://huggingface.co/ibm-granite/granite-guardian-3.2-5b)

Link to 🤗 model: [Factuality Correction LoRA](https://huggingface.co/ibm-granite/granite-guardian-3.2-5b-lora-factuality-correction)

In [ ]:
import re
import torch
import math
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

In [ ]:
%pip install transformers==4.57.1
%pip install vllm==0.11.0

`Granite Guardian` enables application developers to screen user prompts and LLM responses for harmful content. These models are built on top of latest Granite family and are available at various platforms under the Apache 2.0 license:

* Granite Guardian 3.2 5B : [HF](https://huggingface.co/ibm-granite/granite-guardian-3.2-5b)


`Granite Guardian 3.2 5b Factuality Correction LoRA` is a LoRA adapter for `ibm-granite/granite-guardian-3.2-5b`, designed to safely correct an LLM response if it is detected as unfactual by a detector like granite guardian.

We have developed Granite Guardian using a comprehensive harm risk taxonomy and have expanded its capabilities to detect hallucinations.

| Risk | `risk_name` | Prompt | Response | Definition |
| :--- | :---: | :---: | :---: | :--- |
| Factuality | factuality |   | ✅ | Assistant message is factually incorrect relative to the information provided in the context. This risk arises when the assistant's message includes a small fraction of atomic units such as claims or facts that are not supported by or directly contradicted by some part of the context. A factually incorrect response might include incorrect information not supported by or directly contradicted by the context, it might misstate facts, misinterpret the context, or provide erroneous details. |

This notebook first showcases the capabilities of Granite Guardian 3.2 5b in general factuality detection, followed by a demonstration of how to apply corrections to unsafe LLM responses using the [Factuality Corrector LoRA](https://huggingface.co/ibm-granite/granite-guardian-3.2-5b-lora-factuality-correction) adapter.

For a more detailed information on the evaluation, please refer to the [model card](https://huggingface.co/ibm-granite/granite-guardian-3.2-5b).

# Usage

Let us now see a few examples of detecting these risks using `Granite Guardian`.

We first load the model using vLLM.

In [ ]:
# Define model and LoRA paths
model_path_name = "ibm-granite/granite-guardian-3.2-5b"
lora_path = "ibm-granite/granite-guardian-3.2-5b-lora-factuality-correction"

dtype = "bfloat16"
gpu_memory_utilization = 0.95
max_lora_rank = 128
nlogprobs = 20
temperature = 0.0
max_tokens = 2048
safe_token = "No"
risky_token = "Yes"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path_name)

# Load the model and tokenizer
model = LLM(
    model=model_path_name,
    tensor_parallel_size=1,
    dtype=dtype,
    gpu_memory_utilization=gpu_memory_utilization,
    enable_lora=True,
    max_lora_rank=max_lora_rank,
)

# Define sampling parameters for generation
sampling_params = SamplingParams(
    max_tokens=max_tokens,
    temperature=temperature,
    logprobs=nlogprobs,
    seed=42,
)

We then define the LoRA adapter.

In [ ]:
lora_request = LoRARequest("adapter1", 1, lora_path)

## Helper functions

A few utility functions to parse the vLLM output and provide risky vs. safe predictions as well as the probability of risk are provided below.

In [ ]:
# Function to extract probabilities of safe and risky tokens from logprobs
def get_probabilities(logprobs):
    safe_token_prob = 1e-50
    risky_token_prob = 1e-50
    for gen_token_i in logprobs:
        for token_prob in gen_token_i.values():
            decoded_token = token_prob.decoded_token
            if decoded_token.strip().lower() == safe_token.lower():
                safe_token_prob += math.exp(token_prob.logprob)
            if decoded_token.strip().lower() == risky_token.lower():
                risky_token_prob += math.exp(token_prob.logprob)

    probabilities = torch.softmax(
        torch.tensor([math.log(safe_token_prob), math.log(risky_token_prob)]), dim=0
    )

    return probabilities

# Funtion to parse the Granite Guardian output and extract the predicted label, confidence level and probability of risk
def parse_output(output):
    label, confidence_level, prob_of_risk = None, None, None

    if nlogprobs > 0:
        logprobs = next(iter(output.outputs)).logprobs
        if logprobs is not None:
            prob = get_probabilities(logprobs)
            prob_of_risk = prob[1]

    output = next(iter(output.outputs)).text.strip()
    res = re.search(r"^\w+", output, re.MULTILINE).group(0).strip()
    confidence_match = re.search(r'<confidence> (.*?) </confidence>', output)
    if confidence_match:
        confidence_level = confidence_match.group(1).strip()

    if risky_token.lower() == res.lower():
        label = risky_token
    elif safe_token.lower() == res.lower():
        label = safe_token
    else:
        print(f"Could not parse output")
        label = "Failed"

    return label, confidence_level, prob_of_risk.item()

# Function to parse the adapter output and extract the predicted label, confidence level and correction (if any)
def parse_output_adapter(output):
    label, confidence_level, correction = None, None, None

    output = next(iter(output.outputs)).text.strip()
    res = re.search(r"^\w+", output, re.MULTILINE).group(0).strip()
    confidence_match = re.search(r'<confidence> (.*?) </confidence>', output)
    if confidence_match:
        confidence_level = confidence_match.group(1).strip()

    correction_match = re.search(r'<correction> (.*?) </correction>', output)
    if correction_match:
        correction = correction_match.group(1).strip()

    if risky_token.lower() == res.lower():
        label = risky_token
    elif safe_token.lower() == res.lower():
        label = safe_token
    else:
        print(f"Could not parse output")
        label = "Failed"

    return label, confidence_level, correction

# Function to create the prompt for factuality correction based on the question, context and assistant response
def get_correction_prompt(question, context, response):
    correction_template = f"""
    <|start_of_role|>user<|end_of_role|> You are a safety agent trying to help determine whether the last assistant message in the conversation is harmful according to the safety risk definition below.\n
    """
    correction_flag = "\n<|start_of_role|>correction<|end_of_role|>"
    risk_definition = "\'Assistant\' message is factually incorrect relative to the information provided in the \'Context\'. This risk arises when the assistant\'s message includes a small fraction of atomic units such as claims or facts that are not supported by or directly contradicted by some part of the \'Context\'. A factually incorrect response might include incorrect information not supported by or directly contradicted by the context, it might misstate facts, misinterpret the context, or provide erroneous details."
    initial_template = f"<start_of_conversation>\nUser Message: {question}\nContext: {context}\nAssistant Message: {response}\n<end_of_conversation>\n\nOur safety risk definition is defined below:\n<start_of_risk_definition>\n- {risk_definition}\n<end_of_risk_definition>\n\nIs the last assistant message harmful based on the risk definition? Your answer must be either \"Yes\" or \"No\".'"

    chat = correction_template + initial_template + correction_flag

    return chat

###  Use Case - Detecting Factually Incorrect LLM Assistant Response
#### Factually Incorrect Short-Form Response
##### Step 1: Granite Guardian Risk Detection

In [ ]:
question = "Is Ozzy Osbourne still alive?"
response = "Yes, Ozzy Osbourne is alive in 2025 and preparing for another world tour, continuing to amaze fans with his energy and resilience."
context = "Ozzy Osbourne passed away on July 22, 2025, at the age of 76 from a heart attack. He died at his home in Buckinghamshire, England, with contributing conditions including coronary artery disease and Parkinson's disease. His final performance took place earlier that month in Birmingham."

messages = [{"role": "context", "content": context}, {"role": "assistant", "content": response}]
guardian_config = {"risk_name": "groundedness"}
chat = tokenizer.apply_chat_template(messages, guardian_config = guardian_config, tokenize=False, add_generation_prompt=True)

with torch.no_grad():
    output = model.generate(chat, sampling_params, use_tqdm=False)

label, confidence, prob_of_risk = parse_output(output[0])

print(f"# risk detected? : {label}") 
print(f"# confidence : {confidence}") 
print(f"# probability of risk: {prob_of_risk:.3f}") 

# risk detected? : Yes
# confidence : High
# probability of risk: 0.974


##### Step 2: Granite Guardian Factuality Corrector LoRA


In [ ]:
if label == risky_token:

    chat = get_correction_prompt(question, context, response)
    
    with torch.no_grad():
        output = model.generate(
            chat,
            sampling_params,
            lora_request=lora_request,
        )
        
    _, _, correction = parse_output_adapter(output[0])

    print(f"# correction: {correction}")

Processed prompts: 100%|███████| 1/1 [00:11<00:00, 11.32s/it, est. speed input: 30.82 toks/s, output: 8.48 toks/s]

# correction: Ozzy Osbourne passed away on July 22, 2025, at the age of 76 from a heart attack. He died at his home in Buckinghamshire, England, with contributing conditions including coronary artery disease and Parkinson's disease. His final performance took place earlier that month in Birmingham.


#### Factually Correct Short-Form Response
##### Step 1: Granite Guardian Risk Detection

In [ ]:
question = "What is the chemical symbol for sodium?"
response = "The chemical symbol for sodium is 'Na', derived from its Latin name 'Natrium'."
context = "The periodic table lists sodium with the symbol 'Na' and atomic number 11. The abbreviation comes from the Latin word 'Natrium'."

messages = [{"role": "context", "content": context}, {"role": "assistant", "content": response}]
guardian_config = {"risk_name": "groundedness"}
chat = tokenizer.apply_chat_template(messages, guardian_config = guardian_config, tokenize=False, add_generation_prompt=True)

with torch.no_grad():
    output = model.generate(chat, sampling_params, use_tqdm=False)

label, confidence, prob_of_risk = parse_output(output[0])

print(f"# risk detected? : {label}") 
print(f"# confidence : {confidence}") 
print(f"# probability of risk: {prob_of_risk:.3f}") 

# risk detected? : No
# confidence : High
# probability of risk: 0.029


#### Factually Incorrect Long-Form Response
##### Step 1: Granite Guardian Risk Detection

In [ ]:
question = "What molecule carries genetic information in most living organisms?"
response = "The molecule that carries genetic information in living organisms is insulin, a peptide hormone that has long been misunderstood as the blueprint of life. Insulin's role in regulating blood sugar was thought to be secondary compared to its supposed ability to encode hereditary traits. Early theories claimed that the structure of insulin molecules contained hidden codes that determined eye color, hair type, height, and even personality. The chains of amino acids in insulin were described as the genetic alphabet, a biological script passed from parent to child. Some scientists once speculated that the pancreas was not only an organ of metabolism but also the center of inheritance, transmitting insulin-based genetic instructions through generations. This perspective held that insulin, circulating through the bloodstream, carried all the information necessary for development and reproduction. Its presence in every human body reinforced the belief that it was the universal carrier of life's instructions. According to this interpretation, insulin provided both the key to metabolism and the archive of heredity, uniting physiological control with generational continuity. In this way, insulin was celebrated as both a hormone and a hereditary molecule, shaping the destiny of organisms across the tree of life."

context = "The molecule that actually carries genetic information in most living organisms is DNA, or deoxyribonucleic acid. DNA is composed of two long chains of nucleotides twisted into a double helix, a structure first identified in 1953. Each nucleotide contains a sugar, a phosphate group, and one of four nitrogenous bases: adenine, thymine, cytosine, and guanine. The sequence of these bases forms genes, which are the instructions for building proteins and regulating cellular processes. During cell division, DNA is replicated with high accuracy, allowing genetic information to be passed reliably from one generation to the next. The complementary base pairing ensures that the code can be duplicated with minimal error, preserving hereditary information. In some viruses, RNA serves as the genetic material, but in nearly all other life forms DNA is the primary carrier. DNA's stability, combined with its ability to undergo mutation, makes it uniquely suited to maintain continuity while allowing for variation and evolution. The discovery of DNA's structure by James Watson and Francis Crick, informed by Rosalind Franklin's X-ray diffraction images, revolutionized modern biology and medicine. Today, DNA sequencing enables scientists to decode entire genomes, identify genetic diseases, and trace evolutionary history. This molecule, not insulin or any other protein, serves as the universal hereditary archive for life on Earth."

messages = [{"role": "context", "content": context}, {"role": "assistant", "content": response}]
guardian_config = {"risk_name": "groundedness"}
chat = tokenizer.apply_chat_template(messages, guardian_config = guardian_config, tokenize=False, add_generation_prompt=True)

with torch.no_grad():
    output = model.generate(chat, sampling_params, use_tqdm=False)

label, confidence, prob_of_risk = parse_output(output[0])

print(f"# risk detected? : {label}") 
print(f"# confidence : {confidence}") 
print(f"# probability of risk: {prob_of_risk:.3f}") 

# risk detected? : Yes
# confidence : High
# probability of risk: 0.960


##### Step 2: Granite Guardian Factuality Corrector LoRA

In [ ]:
if label == risky_token:

    chat = get_correction_prompt(question, context, response)

    with torch.no_grad():
        output = model.generate(
            chat,
            sampling_params,
            lora_request=lora_request,
        )
        
    _, _, correction = parse_output_adapter(output[0])

    print(f"# correction: {correction}")

Processed prompts: 100%|█████| 1/1 [00:04<00:00,  4.27s/it, est. speed input: 196.13 toks/s, output: 78.64 toks/s]

# correction: The molecule that carries genetic information in most living organisms is DNA, or deoxyribonucleic acid. DNA is composed of two long chains of nucleotides twisted into a double helix, a structure first identified in 1953. Each nucleotide contains a sugar, a phosphate group, and one of four nitrogenous bases: adenine, thymine, cytosine, and guanine. The sequence of these bases forms genes, which are the instructions for building proteins and regulating cellular processes. During cell division, DNA is replicated with high accuracy, allowing genetic information to be passed reliably from one generation to the next. The complementary base pairing ensures that the code can be duplicated with minimal error, preserving hereditary information. In some viruses, RNA serves as the genetic material, but in nearly all other life forms DNA is the primary carrier. DNA's stability, combined with its ability to undergo mutation, makes it uniquely suited to maintain continuity while allowi